In [167]:
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV 
from sklearn.linear_model import LogisticRegression 
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline 

In [81]:
#import nltk 
#nltk.download('stopwords') 

In [49]:
df_train = pd.read_csv('train.csv',index_col='Id')

In [50]:
df_train

,review,y
Id,,
0,"Don't waste 90 minutes of your time on ""Fast F...",0
1,Even with a cast that boasts such generally re...,0
2,This is a great movie. In the same genre of th...,1
3,"I'm sick of people whining about Ewoks! True, ...",1
4,This film was shot on location in Gerard Garde...,1
...,...,...
24995,An intriguing premise of hand-drawn fantasy co...,0
24996,I didn't expect much when I first saw the DVD ...,1
24997,"Part II or formerly known as GUERILLA, is also...",1


#### Классы сбалансированны

In [157]:
df_train['y'].value_counts()

y
0    12500
1    12500
Name: count, dtype: int64

In [219]:
df_test = pd.read_csv('test.csv',index_col='Id')

In [ ]:
df

In [7]:
df_train['review'][0]

'Don\'t waste 90 minutes of your time on "Fast Food, Fast Women." It\'s annoyingly episodic script with three story lines patched together is laughably bad due to predictable writing, horrific acting, and even bad music. I found the anorexic main character upsetting to watch every time she was on screen. SHE needs the fast food.<br /><br />Spend the 90 minutes you\'d devote to this turkey doing something more exciting...like trimming your toenails. You\'d have more entertainment value.<br /><br />The only redeeming thing about this film is Louise Lasser, but she deserves much better than this tired script. It\'s as impotent as the elder guy she courts in the movie.<br /><br />VIEWER BEWARE!'

#### Удалим ненужные символы

In [8]:
def preprocess(text):
    text = re.sub(r"<[^>]*>","",text)
    text = re.sub(r"[\W]+",' ',text.lower())
    return text

In [80]:
preprocess(df_train['review'][0])

'don t waste 90 minutes of your time on fast food fast women it s annoyingly episodic script with three story lines patched together is laughably bad due to predictable writing horrific acting and even bad music i found the anorexic main character upsetting to watch every time she was on screen she needs the fast food spend the 90 minutes you d devote to this turkey doing something more exciting like trimming your toenails you d have more entertainment value the only redeeming thing about this film is louise lasser but she deserves much better than this tired script it s as impotent as the elder guy she courts in the movie viewer beware '

In [9]:
df_train['review'] = df_train['review'].apply(preprocess)

In [13]:
def tokenizer(text): 
    return text.split()

#### Стэмминг + удаление стоп-слов

In [138]:
from nltk.stem.porter import PorterStemmer 
from nltk.corpus import stopwords
stop = stopwords.words('english')
porter = PorterStemmer() 
def tokenizer_porter(text): 
    words = [porter.stem(word) for word in text.split()] 
    nostop = [word for word in words if word not in stop]
    return " ".join(nostop)

In [139]:
clear = df_train['review'].apply(preprocess)

In [140]:
clear = clear.apply(tokenizer_porter)

In [274]:
p = CountVectorizer(lowercase=False,stop_words=None,tokenizer=None)

In [275]:
X_train = p.fit_transform(clear)

In [281]:
p = CountVectorizer()

In [282]:
X_train = p.fit_transform(df_train['review'])

In [283]:
clf = LogisticRegression(n_jobs=-1)

In [284]:
clf.fit(X_train,df_train['y'])

LogisticRegression(n_jobs=-1)

In [285]:
pred = clf.predict_proba(p.transform(df_test['review']))[:,1]

In [286]:
pred

array([0.00571137, 0.00376727, 0.89272935, ..., 0.01245046, 0.99998711,
       0.94484078])

##### Самый простой подход к решению задачи достигает 0.85 на kaggle (метрика - ROC-AUC)
##### Также попробовал не делать стэмминг и удаление стоп слов вовсе, результат поднялся на 0.93

##### Автоматизируем процесс

In [312]:
X_train, X_test, y_train, y_test = train_test_split(df_train['review'],df_train['y'], test_size=0.33, random_state=42)

In [291]:
pipeline = Pipeline([('vectorizer',CountVectorizer()),('classfier',LogisticRegression(n_jobs=-1))])

In [292]:
pipeline

Pipeline(steps=[('vectorizer', CountVectorizer()),
                ('classfier', LogisticRegression(n_jobs=-1))])

In [293]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('vectorizer', CountVectorizer()),
                ('classfier', LogisticRegression(n_jobs=-1))])

In [294]:
pred = pipeline.predict_proba(X_test)[:,1]

In [296]:
roc_auc_score(y_test,pred)

0.9426924501918791

In [319]:
pipelineTFIDF = Pipeline([('vectorizer',TfidfVectorizer(stop_words=["english"])),('classfier',LogisticRegression(n_jobs=-1))])

In [320]:
pipelineTFIDF.fit(X_train,y_train)

Pipeline(steps=[('vectorizer', TfidfVectorizer(stop_words=['english'])),
                ('classfier', LogisticRegression(n_jobs=-1))])

In [321]:
pred = pipelineTFIDF.predict_proba(df_test['review'])[:,1]

#### tf-idf достиг 0.95

In [299]:
for_kaggle = pipeline.predict_proba(df_test['review'])[:,1]

In [300]:
def submission(predictions):
    sub = pd.DataFrame(predictions,columns=['y'])
    sub.index.name = 'Id'
    sub.to_csv("mysubmission.csv")

In [301]:
submission(for_kaggle)

In [322]:
submission(pred)